# 🔬 SciMSim — Scientific Scenario Simulation

A modular LLM-driven simulation engine for exploring hypothetical futures of scientific fields.

**How it works:**
- A population of researcher *personas* is generated from your config.
- Each timestep, active researchers read papers from the growing corpus and write new ones.
- Papers cite prior work; impact scores propagate through the corpus.
- External shocks can be injected at any timestep.
- After the run, you get a readable timeline, top papers, and keyword evolution.

This is **scenario exploration**, not prediction — the goal is to generate internally consistent,
plausible alternative futures and stress-test assumptions.

## 0. Install & Import

In [ ]:
# Install dependencies if needed
# !pip install anthropic          # for Claude backend
# !pip install openai             # for GPT backend

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from scimsim import (
    SimulationConfig,
    ExternalShock,
    SchoolOfThought,
    IncentiveStructure,
    run_simulation,
    print_scenario_summary,
    timeline_view,
    export_corpus_md,
)

## 1. API Key

Set your API key here. It is never stored or logged.

In [ ]:
# Choose your provider
PROVIDER = "anthropic"   # or "openai"

# Paste your key here, or load from environment
import os
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
# API_KEY = os.environ.get("OPENAI_API_KEY", "")  # if using OpenAI

if not API_KEY:
    API_KEY = input("Paste your API key: ").strip()

print(f"Provider: {PROVIDER} | Key set: {'yes' if API_KEY else 'NO'}")

## 2. Configure Your Simulation

All parameters are here. Change them freely — each combination produces a different scenario.

| Parameter | What it controls |
|-----------|------------------|
| `domain` | The scientific field being simulated |
| `seed_idea` | The founding idea the simulation bootstraps from |
| `num_timesteps` | How many years to simulate |
| `papers_per_timestep` | Research output per year (more = richer but slower) |
| `num_researchers` | Size of the simulated community |
| `epistemic_openness` | 0 = paradigm-locked, 1 = freely cross-pollinating |
| `incentive_structure` | academic / industry / balanced |
| `wildcard` | 0 = conservative ideas, 1 = weird/unexpected |
| `shocks` | Discrete events injected at specific timesteps |

In [ ]:
cfg = SimulationConfig(
    # ── Core setup ─────────────────────────────────────────────────────
    domain       = "artificial intelligence",
    seed_idea    = "large language models can reason across domains when scaled",
    start_year   = 2025,
    num_timesteps       = 5,   # simulated years
    papers_per_timestep = 3,   # papers generated per year

    # ── Community sociology ────────────────────────────────────────────
    num_researchers = 12,
    school_distribution = {
        SchoolOfThought.DOMINANT:    0.4,   # mainstream scaling researchers
        SchoolOfThought.CHALLENGER:  0.2,   # heterodox critics
        SchoolOfThought.APPLIED:     0.3,   # engineers / deployment focus
        SchoolOfThought.THEORETICAL: 0.1,   # formal / mathematical
    },
    epistemic_openness = 0.5,   # 0=stay in paradigm, 1=freely cross

    # ── Incentives ─────────────────────────────────────────────────────
    incentive_structure = IncentiveStructure.BALANCED,

    # ── Wildcards ──────────────────────────────────────────────────────
    wildcard = 0.35,   # 0=conservative, 1=unexpected ideas

    # ── External shocks ────────────────────────────────────────────────
    # Inject discrete events at specific timesteps (0-indexed).
    # Comment out to run without shocks.
    shocks = [
        ExternalShock(
            timestep=2,
            description=(
                "A major safety incident involving a deployed LLM causes "
                "public backlash and forces the community to prioritize "
                "alignment and interpretability research."
            ),
        ),
        ExternalShock(
            timestep=4,
            description=(
                "Compute costs drop 10x due to new hardware. "
                "Small labs can now run experiments previously only possible at frontier labs."
            ),
            affects_schools=[SchoolOfThought.CHALLENGER, SchoolOfThought.APPLIED],
        ),
    ],

    # ── LLM backend ────────────────────────────────────────────────────
    llm_provider = PROVIDER,
    api_key      = API_KEY,
    # llm_model  = "claude-haiku-4-5"  # override model if desired
)

## 3. Run the Simulation

In [ ]:
state = run_simulation(cfg, verbose=True)

## 4. Explore the Results

In [ ]:
# High-level stats and top papers
print_scenario_summary(state)

In [ ]:
# Readable year-by-year timeline with abstracts
timeline_view(state)

In [ ]:
# Export the full corpus as a Markdown file
export_corpus_md(state, path="my_scenario.md")

## 5. Compare Two Scenarios

The real power of the engine is running the same seed under different parameters
and comparing where the field ends up.

In [ ]:
# Scenario A: industry-driven, conservative, no shocks
cfg_a = SimulationConfig(
    domain="artificial intelligence",
    seed_idea="large language models can reason across domains when scaled",
    start_year=2025,
    num_timesteps=4,
    papers_per_timestep=2,
    incentive_structure=IncentiveStructure.INDUSTRY,
    wildcard=0.1,
    epistemic_openness=0.2,
    llm_provider=PROVIDER,
    api_key=API_KEY,
)

# Scenario B: academic, high wildcard, major shock
cfg_b = SimulationConfig(
    domain="artificial intelligence",
    seed_idea="large language models can reason across domains when scaled",
    start_year=2025,
    num_timesteps=4,
    papers_per_timestep=2,
    incentive_structure=IncentiveStructure.ACADEMIC,
    wildcard=0.8,
    epistemic_openness=0.9,
    shocks=[ExternalShock(timestep=1, description="A breakthrough in neuromorphic computing makes transformer architectures obsolete.")],
    llm_provider=PROVIDER,
    api_key=API_KEY,
)

print("Running Scenario A (industry, conservative)...")
state_a = run_simulation(cfg_a, verbose=False)

print("\nRunning Scenario B (academic, wild, shocked)...")
state_b = run_simulation(cfg_b, verbose=False)

print("\n" + "="*70)
print("SCENARIO A — Top papers:")
for p in sorted(state_a.corpus, key=lambda p: p.impact_score, reverse=True)[:3]:
    print(f"  [{p.impact_score:.1f}] {p.title}")

print("\nSCENARIO B — Top papers:")
for p in sorted(state_b.corpus, key=lambda p: p.impact_score, reverse=True)[:3]:
    print(f"  [{p.impact_score:.1f}] {p.title}")

## 6. Inspect Individual Papers

In [ ]:
# Browse all papers from the main run
for p in state.corpus:
    if p.timestep < 0:
        continue  # skip seed
    print(f"Year {cfg.year_of(p.timestep)} | [{p.school.value}] | Impact {p.impact_score}")
    print(f"  {p.title}")
    print(f"  {p.abstract[:200]}...")
    print()